# TransAxx sweep: 2 datasets x 3 models x 6 multipliers x (eval-only, retrained)

GPU mirror of `1_run_sweep_adapt.ipynb`. Same matrix, same CSV schema, same per-run confusion-matrix artifact layout, just under `results/` of this container (mapped to `docker/results/transaxx/` on the host).

Models come from `chenyaofo/pytorch-cifar-models` (CIFAR-10 and CIFAR-100 pretrained variants). `replace_conv_layers` from `classification.utils` wraps every `nn.Conv2d` in `AdaptConv2D`, the GPU approximate-conv layer.

Re-running is safe: rows already in the CSV are skipped. To recompute everything from scratch (e.g. after changing the recipe or LUTs), set `FRESH = True` in the config cell.

In [ ]:
import csv, gc, os, time, uuid
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms as T
from torchvision.datasets import CIFAR10, CIFAR100
from tqdm import tqdm

from classification.utils import (replace_conv_layers, collect_stats, compute_amax,
                                   AdaptConv2D)
from classification.train import train_one_epoch

device = "cuda"
assert torch.cuda.is_available(), "CUDA not available — check nvidia-container-toolkit / CDI override"

## Sweep config

**Models / datasets / parameters are set in `0_configure_sweeps.ipynb`** (edit there, Run All,
then re-run this notebook, since the cell below just reads its `sweep_config.json`). Only `FRESH`
lives here, because it deletes results. Results path is the `results/` volume mount, which
the compose file maps back to the host's `docker/results/transaxx/`.

In [ ]:
import json as _json

# What to sweep comes from 0_configure_sweeps' manifest (EDIT THERE, not here).
# The literals below are only fallbacks for when the manifest hasn't been written yet.
_cfg_all = _json.loads(Path("sweep_config.json").read_text()) \
           if Path("sweep_config.json").exists() else {}
_cfg = _cfg_all.get("transaxx", {})
DATASETS       = _cfg.get("datasets", ["cifar10", "cifar100"])
MODELS         = _cfg.get("models", ["resnet20", "resnet56", "vgg11_bn"])
RETRAIN_EPOCHS = _cfg.get("retrain_epochs", 3)
BATCH_SIZE     = _cfg.get("batch_size", 128)
CALIB_BATCHES  = _cfg.get("calib_batches", 2)

# Multipliers come from 0_build_multipliers' manifest (define once, sweep everywhere).
# mul8s_exact is the generator's exact twin of the mul8s_acc baseline, so we keep acc
# and drop the duplicate. Falls back to the fixed list if the manifest is not built yet.
_manifest = Path("../luts/multipliers.json")
if _manifest.exists():
    _built = _json.loads(_manifest.read_text())["multipliers"]
    MULTS = [m for m in _built if m != "mul8s_exact"] + ["mul8s_acc"]
else:
    MULTS = [f"mul8s_fpga_ish_{i}" for i in range(1, 6)] + ["mul8s_acc"]

# FRESH = False -> resume: rows already in runs.csv are skipped, so you can
#                  interrupt and re-run safely. NOTE: it only checks whether a
#                  row exists, not whether it is up to date -- if you change the
#                  recipe / LUTs / calibration it will NOT recompute existing rows.
# FRESH = True  -> wipe runs.csv + confusion_matrices/ (and legacy runs/) first and
#                  recompute the whole matrix from scratch.
FRESH = False

# Jupyter roots at /work, and this notebook lives in /work/notebooks, so results go to
# /work/results/transaxx (compose mounts ./results -> /work/results).
RESULTS_DIR = Path("../results/transaxx")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH = RESULTS_DIR / "runs.csv"
COLS = ["run_id", "timestamp", "dataset", "model", "multiplier",
        "retrained_epochs", "top1", "top5", "n_samples", "runtime_s", "notes"]

if FRESH:
    import shutil
    CSV_PATH.unlink(missing_ok=True)
    for _d in (RESULTS_DIR / "confusion_matrices", RESULTS_DIR / "runs"):
        shutil.rmtree(_d, ignore_errors=True)
    print("FRESH=True -> cleared runs.csv, confusion_matrices/ and legacy runs/")

print(f"Matrix size: {len(DATASETS)} x {len(MODELS)} x {len(MULTS)} x 2 = "
      f"{len(DATASETS)*len(MODELS)*len(MULTS)*2} rows")

## Data loaders

TransAxx ships a CIFAR-10 loader in `classification.utils` but no CIFAR-100, so we write both here for symmetry with the AdaPT notebook. Calibration set is every 10th sample of train (10% subset).

In [ ]:
CIFAR_INFO = {
    "cifar10":  {"mean": (0.4914, 0.4822, 0.4465), "std": (0.2471, 0.2435, 0.2616),
                  "cls": CIFAR10,  "n_classes": 10},
    "cifar100": {"mean": (0.5071, 0.4867, 0.4408), "std": (0.2675, 0.2565, 0.2761),
                  "cls": CIFAR100, "n_classes": 100},
}

def make_loaders(dataset, batch_size=BATCH_SIZE):
    info = CIFAR_INFO[dataset]
    tf = T.Compose([T.ToTensor(), T.Normalize(info["mean"], info["std"])])
    root = f"/workspace/datasets/{dataset}_data"
    test = info["cls"](root=root, train=False, download=True, transform=tf)
    train = info["cls"](root=root, train=True, download=True, transform=tf)
    val = DataLoader(test, batch_size=batch_size, num_workers=8,
                     shuffle=False, drop_last=True, pin_memory=True)
    evens = list(range(0, len(train), 10))
    cal = DataLoader(torch.utils.data.Subset(train, evens),
                     batch_size=batch_size, shuffle=False,
                     num_workers=8, drop_last=True, pin_memory=True)
    return val, cal

## Model builder + conv wrapping

`install_multiplier` calls `replace_conv_layers` with `initialize=True` every time, which is what the upstream `custom_test.ipynb` does. That rewraps each conv with a fresh `AdaptConv2D` carrying the new `axx_mult`. The JIT compile is cached per multiplier name.

In [ ]:
def build_model(name, dataset):
    hub_id = f"{dataset}_{name}"
    return torch.hub.load("chenyaofo/pytorch-cifar-models", hub_id, pretrained=True).to(device)

def conv_layer_count(model):
    return sum(1 for n, m in model.named_modules()
               if (isinstance(m, nn.Conv2d) or isinstance(m, AdaptConv2D))
               and "head" not in n and "reduction" not in n)

def install_multiplier(model, mult, n_layers):
    axx_list = [{"axx_mult": mult, "axx_power": 1.0,
                  "quant_bits": 8, "fake_quant": False}] * n_layers
    replace_conv_layers(model, AdaptConv2D, axx_list, 0, 0,
                         layer_count=[0], returned_power=[0],
                         initialize=True)

## Calibration, evaluation, finetune helpers

`collect_stats` / `compute_amax` come from `classification.utils` and take a `device=` kwarg. Evaluation also accumulates the per-(true, predicted) confusion matrix in the same pass.

In [ ]:
def calibrate(model, calib_loader):
    with torch.no_grad():
        collect_stats(model, calib_loader, num_batches=CALIB_BATCHES, device=device)
        compute_amax(model, method="percentile", percentile=99.99, device=device)

def evaluate(model, loader, n_classes):
    model.eval()
    correct = top5 = total = 0
    cm = np.zeros((n_classes, n_classes), dtype=np.int64)
    t0 = time.time()
    with torch.no_grad():
        for img, lbl in tqdm(loader, total=len(loader), leave=False):
            img, lbl = img.to(device), lbl.to(device)
            out = model(img)
            _, p5 = out.topk(5, dim=1)
            p1 = p5[:, 0]
            total += lbl.size(0)
            correct += (p1 == lbl).sum().item()
            top5 += (p5 == lbl.unsqueeze(1)).any(dim=1).sum().item()
            for t, p in zip(lbl.cpu().numpy(), p1.cpu().numpy()):
                cm[t, p] += 1
    dt = time.time() - t0
    return 100 * correct / total, 100 * top5 / total, total, dt, cm

def finetune(model, loader, epochs):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=1e-4)
    for e in range(epochs):
        train_one_epoch(model, criterion, optimizer, loader, device, e, 50)

In [ ]:
def existing_keys():
    if not CSV_PATH.exists():
        return set()
    df = pd.read_csv(CSV_PATH)
    return {tuple(r) for r in df[["dataset", "model", "multiplier", "retrained_epochs"]]
                                  .itertuples(index=False, name=None)}

def append_row(row):
    new_file = not CSV_PATH.exists()
    with CSV_PATH.open("a", newline="") as f:
        w = csv.writer(f)
        if new_file:
            w.writerow(COLS)
        w.writerow([row.get(c, "") for c in COLS])

def cm_path(dataset, model_name, mult, retrained_epochs):
    """Human-readable, dataset-foldered confusion-matrix path, e.g.
    results/confusion_matrices/cifar100/resnet20_fpga_ish_1_e0.npy."""
    short = mult.replace("mul8s_", "")
    d = RESULTS_DIR / "confusion_matrices" / dataset
    d.mkdir(parents=True, exist_ok=True)
    return d / f"{model_name}_{short}_e{retrained_epochs}.npy"

def record(dataset, model_name, mult, retrained_epochs, top1, top5, n, dt, cm):
    rid = uuid.uuid4().hex[:12]
    np.save(cm_path(dataset, model_name, mult, retrained_epochs), cm)
    append_row({
        "run_id": rid,
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
        "dataset": dataset, "model": model_name, "multiplier": mult,
        "retrained_epochs": retrained_epochs,
        "top1": f"{top1:.4f}", "top5": f"{top5:.4f}",
        "n_samples": n, "runtime_s": f"{dt:.1f}", "notes": "",
    })
    return rid

## Sweep

Same shape as the AdaPT notebook. The first multiplier per (dataset, model) JIT-compiles the GPU kernel (~30s). Subsequent multipliers recompile per name and rewrap the convs (~30s upstream-observed swap time).

Rows already present in `results/runs.csv` are skipped, so interrupted runs resume cleanly.

In [ ]:
done = existing_keys()
print(f"{len(done)} rows already in CSV; will skip those.\n")

for dataset in DATASETS:
    val, cal = make_loaders(dataset)
    n_cls = CIFAR_INFO[dataset]["n_classes"]
    for model_name in MODELS:
        for mult in MULTS:
            need_eval    = (dataset, model_name, mult, 0) not in done
            need_retrain = RETRAIN_EPOCHS > 0 and \
                            (dataset, model_name, mult, RETRAIN_EPOCHS) not in done
            if not (need_eval or need_retrain):
                print(f"[skip] {dataset}/{model_name}/{mult} — both rows present")
                continue

            print(f"\n=== {dataset} / {model_name} / {mult} ===")
            m = build_model(model_name, dataset)
            n_layers = conv_layer_count(m)
            install_multiplier(m, mult, n_layers)
            calibrate(m, cal)

            if need_eval:
                top1, top5, n, dt, cm = evaluate(m, val, n_cls)
                rid = record(dataset, model_name, mult, 0, top1, top5, n, dt, cm)
                print(f"  eval-only      top1={top1:.4f}  top5={top5:.4f}  ({dt:.1f}s)  run_id={rid}")

            if need_retrain:
                finetune(m, cal, RETRAIN_EPOCHS)
                top1, top5, n, dt, cm = evaluate(m, val, n_cls)
                rid = record(dataset, model_name, mult, RETRAIN_EPOCHS,
                             top1, top5, n, dt, cm)
                print(f"  retrained({RETRAIN_EPOCHS}ep) top1={top1:.4f}  top5={top5:.4f}  ({dt:.1f}s)  run_id={rid}")

            del m
            torch.cuda.empty_cache()
            gc.collect()

from IPython.display import FileLink, display

print("Done.")
print(f"Results saved under: {RESULTS_DIR.resolve()}\n")

# Live inventory of what the results folder holds right now.
n_rows = sum(1 for _ in open(CSV_PATH)) - 1
n_cm = len(list((RESULTS_DIR / "confusion_matrices").glob("*/*.npy")))
print(f"  runs.csv                                {n_rows} rows -- one per (dataset, model, multiplier, retrained/not)")
print(f"  confusion_matrices/<dataset>/<..>.npy   {n_cm} files  -- per-run confusion matrix (loaded by 3_multiplier_deepdive.ipynb)")

# Clickable link to the CSV, openable straight from this notebook.
print("\nOpen the results CSV:")
display(FileLink(str(CSV_PATH)))
pd.read_csv(CSV_PATH).tail(15)